In [2]:
# !pip install torchvision

In [1]:
import os
import torch
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms
from PIL import Image

In [2]:
#image load => transform => dataset of all images
class ImageProcessor:
    def __init__(self,root_dir_path,transformation=None):
        self.root_dir_path = root_dir_path
        self.transformation = transformation


        #list of path for all images
        self.all_img_path = [os.path.join(root_dir_path) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_path)

    def __getitem__(self,idx):
        img_path = self.all_img_path[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [3]:
root_dir_path = "./img_align_celeba"

transformation = transforms.Compose([
    transforms.CenterCrop(178), #178x256 => 178 x 178
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))   ]
)

In [4]:
#create dataset
dataset = ImageProcessor(root_dir_path,transformation)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [5]:
#create dataloader
dataloader = DataLoader(
    dataset,
    batch_size = 128,
    shuffle = True
)

# Generator Network

In [6]:
import torch.nn as nn
import torch.optim as optim 
import numpy as np

In [7]:
class Generator(nn.Module):
    def __init__(self,z_dim=100,img_channels=3): #3 is for rgb
        super(Generator,self).__init__()

        #fully connected (dense) layers
        self.model = nn.Sequential(
            nn.Linear(z_dim,256),#100 =>256
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,1024),
            nn.ReLU(),


            nn.Linear(1024,64*64* img_channels),
            nn.Tanh() #convert pixels to  [-1,1] for similar attribute as real img
            
        )
        
    def forward(self,z):
        img = self.model(z)
        img = img.view(img.size(0),3,64,64)
        return img

# Discriminator Network

In [8]:
class Discriminator(nn.Module):
    def __init__(self,img_channels=3):
        super(Discriminator,self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(), #4d tensor ==> 1d

            nn.Linear(img_channels*64*64,1024),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024,512),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(512,256),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(256,1),
            nn.Sigmoid() #prob of being real/fake
            
        )

    def forward(self,img):
        return self.model(img)

In [9]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002,betas=(0.5,0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(),lr=0.0002,betas=(0.5,0.999))

In [10]:
import torch
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [11]:
generator = generator.to(device)
discriminator = discriminator.to(device)

In [12]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.9.1
None
False


AssertionError: Torch not compiled with CUDA enabled